## B3 COTAHIST Options Parser

Parses the B3 historical quotes file (COTAHIST_*.TXT) and extracts
options data with configurable filters.

Usage:
    python b3_options_parser.py

Or import and call parse_cotahist() directly in a notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import brentq
from scipy.stats import norm
from bcb import sgs
import warnings

warnings.filterwarnings("ignore")


# ── Layout constants (1-indexed in spec, converted to 0-indexed slices) ──────

LAYOUT = {
    "tipreg": (0, 2),  # record type: '01' = data
    "date": (2, 10),  # AAAAMMDD
    "bdi": (10, 12),  # 78 = call, 82 = put
    "ticker": (12, 24),  # option ticker (encodes underlying + strike code)
    "tpmerc": (24, 27),  # market type: 070 = calls, 080 = puts
    "nomres": (27, 39),  # company short name
    "especi": (39, 49),  # specification
    "modref": (52, 56),  # currency
    "open": (56, 69),  # (11)V99
    "high": (69, 82),  # (11)V99
    "low": (82, 95),  # (11)V99
    "avg": (95, 108),  # (11)V99
    "close": (108, 121),  # (11)V99  ← primary price
    "bid": (121, 134),  # best bid (11)V99
    "ask": (134, 147),  # best ask (11)V99
    "n_trades": (147, 152),  # N(05)
    "qty": (152, 170),  # N(18) total quantity
    "volume": (170, 188),  # (16)V99
    "strike": (188, 201),  # PREEXE (11)V99
    "expiry": (202, 210),  # DATVEN AAAAMMDD
    "isin": (230, 242),  # CODISI
}

In [2]:
# Helper functions


# ── Price scaling ─────────────────────────────────────────────────────────────
def _parse_price(raw: str) -> float:
    """Convert (11)V99 fixed-point string to float. Returns NaN on failure."""
    raw = raw.strip()
    if not raw or raw == "0" * len(raw):
        return np.nan
    try:
        return int(raw) / 100.0
    except ValueError:
        return np.nan


def _parse_volume(raw: str) -> float:
    """Convert (16)V99 fixed-point string to float."""
    raw = raw.strip()
    if not raw or raw == "0" * len(raw):
        return np.nan
    try:
        return int(raw) / 100.0
    except ValueError:
        return np.nan


def _parse_qty(raw: str) -> int:
    """Parse N(18) integer quantity."""
    raw = raw.strip()
    try:
        return int(raw)
    except ValueError:
        return 0


# ── Black-Scholes utilities ───────────────────────────────────────────────────
def bs_price(
    S: float, K: float, T: float, r: float, sigma: float, option_type: str
) -> float:
    """Black-Scholes price for European option."""
    if T <= 0 or sigma <= 0:
        return max(0.0, (S - K) if option_type == "call" else (K - S))
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def implied_vol(
    market_price: float,
    S: float,
    K: float,
    T: float,
    r: float,
    option_type: str,
    low: float = 1e-6,
    high: float = 10.0,
) -> float:
    """
    Back-solve implied volatility from market price via Brent's method.
    Returns NaN if solution not found.
    """
    if market_price <= 0 or S <= 0 or K <= 0 or T <= 0:
        return np.nan
    try:
        intrinsic = max(0.0, (S - K) if option_type == "call" else (K - S))
        if market_price <= intrinsic:
            return np.nan
        f = lambda sigma: bs_price(S, K, T, r, sigma, option_type) - market_price
        if f(low) * f(high) > 0:
            return np.nan
        return brentq(f, low, high, xtol=1e-6, maxiter=200)
    except Exception:
        return np.nan


def compute_greeks(
    S: float, K: float, T: float, r: float, sigma: float, option_type: str
) -> dict:
    """Compute BS Greeks given IV. Returns dict with delta, gamma, theta, vega."""
    if T <= 0 or sigma <= 0 or S <= 0:
        return {"delta": np.nan, "gamma": np.nan, "theta": np.nan, "vega": np.nan}
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf_d1 = norm.pdf(d1)
    gamma = pdf_d1 / (S * sigma * np.sqrt(T))
    vega = S * pdf_d1 * np.sqrt(T) / 100  # per 1% vol move
    if option_type == "call":
        delta = norm.cdf(d1)
        theta = (
            -S * pdf_d1 * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        ) / 365
    else:
        delta = norm.cdf(d1) - 1
        theta = (
            -S * pdf_d1 * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        ) / 365
    return {"delta": delta, "gamma": gamma, "theta": theta, "vega": vega}

In [3]:
# ── Main parser ───────────────────────────────────────────────────────────────
def parse_cotahist(
    filepath: str | Path,
    # ── Filters ───────────────────────────────────────────────────────────────
    underlying_names: list[str] | None = None,  # e.g. ['BOVA', 'PETR']
    option_types: list[str] | None = None,  # ['call', 'put'] or None = both
    min_trades: int = 10,  # minimum daily trades
    min_volume: float = 1000.0,  # minimum daily financial volume (BRL)
    min_open_interest: int = 0,  # minimum qty (proxy for OI)
    min_price: float = 0.01,  # filter near-zero prices
    max_moneyness: float = 0.30,  # |S-K|/S max (filter deep OTM)
    min_dte: int = 7,  # min days to expiry
    max_dte: int = 90,  # max days to expiry
    # ── Greeks computation ────────────────────────────────────────────────────
    compute_greeks_flag: bool = True,
    risk_free_rate: float = 0.1075,  # fallback scalar if risk_free_rates not provided
    risk_free_rates: (
        pd.DataFrame | None
    ) = None,  # date→annualized rate (from fetch_cdi_rates)
    underlying_prices: pd.DataFrame | None = None,  # optional: date→price map
    # ── Output ────────────────────────────────────────────────────────────────
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Parse B3 COTAHIST fixed-width file and return a clean options DataFrame.

    Parameters
    ----------
    filepath : path to COTAHIST_*.TXT file
    underlying_names : list of substrings to match in NOMRES field.
                       e.g. ['BOVA', 'PETR', 'VALE'] — case insensitive.
                       None = all underlyings.
    option_types : ['call'], ['put'], or None for both.
    min_trades : minimum number of trades in the session.
    min_volume : minimum financial volume in BRL.
    min_open_interest : minimum quantity (QUATOT) — crude OI proxy.
    min_price : minimum option close price (filters worthless contracts).
    max_moneyness : maximum |S-K|/S. Filters deep OTM/ITM.
                    Only applied when underlying prices are available.
    min_dte : minimum days to expiration.
    max_dte : maximum days to expiration.
    compute_greeks_flag : whether to compute IV and Greeks.
    risk_free_rate : annualized fallback rate used when risk_free_rates is None
                     or a date has no CDI observation (e.g. holidays).
    risk_free_rates : DataFrame with columns ['date', 'risk_free_rate'] as
                      returned by fetch_cdi_rates(). When provided, each row
                      gets the CDI rate for its trading date; the scalar
                      risk_free_rate is only used as a fill for missing dates.
    underlying_prices : DataFrame with columns ['date', 'underlying', 'price'].
                        If None, IV/Greeks will be computed using avg price as
                        a rough proxy (less accurate but still useful for
                        preliminary analysis).
    verbose : print progress and summary stats.

    Returns
    -------
    pd.DataFrame with parsed and filtered options data.
    """
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    rows = []
    skipped_filter = 0
    skipped_parse = 0
    total_options = 0

    if verbose:
        print(f"Parsing {filepath.name}...")

    with open(filepath, encoding="latin-1") as f:
        for line in f:
            line = line.rstrip("\n")

            # ── Only data records ─────────────────────────────────────────────
            if len(line) < 245:
                continue
            if line[0:2] != "01":
                continue

            # ── Only options (BDI 78 = call, 82 = put) ───────────────────────
            bdi = line[10:12].strip()
            if bdi not in ("78", "82"):
                continue

            option_type = "call" if bdi == "78" else "put"
            total_options += 1

            # ── Option type filter ────────────────────────────────────────────
            if option_types and option_type not in option_types:
                skipped_filter += 1
                continue

            # ── Underlying name filter ────────────────────────────────────────
            nomres = line[27:39].strip().upper()
            if underlying_names:
                match = any(u.upper() in nomres for u in underlying_names)
                if not match:
                    skipped_filter += 1
                    continue

            # ── Clean underlying name for matching with equity prices ─────────
            # Remove suffixes like "/ED", "/ATZ", "FM/" etc. to match equity NOMRES
            # Examples: "PETRE    /ED" -> "PETRE", "PETR  FM/EDJ" -> "PETR"
            clean_nomres = nomres.split()[0] if " " in nomres else nomres
            clean_nomres = (
                clean_nomres.split("/")[0] if "/" in clean_nomres else clean_nomres
            )
            # Special cases: map option abbreviations to full equity names
            # PETR options: PETRA, PETRB, ..., PETRX all map to PETROBRAS
            # BOVA calls use NOMRES "BOVA"; puts use "BOVAE" — both startswith("BOVA")
            if clean_nomres.startswith("PETR"):
                clean_nomres = "PETROBRAS"
            elif clean_nomres.startswith("BOVA") or clean_nomres == "ISHARES":
                # BOVA options map to BOVA11 equity which has NOMRES "ISHARES BOVA"
                clean_nomres = "ISHARES BOVA"

            # ── Parse prices ──────────────────────────────────────────────────
            try:
                close = _parse_price(line[108:121])
                strike = _parse_price(line[188:201])
                bid = _parse_price(line[121:134])
                ask = _parse_price(line[134:147])
                open_ = _parse_price(line[56:69])
                high = _parse_price(line[69:82])
                low = _parse_price(line[82:95])
                avg = _parse_price(line[95:108])
                volume = _parse_volume(line[170:188])
                qty = _parse_qty(line[152:170])
                n_trades = int(line[147:152].strip() or 0)
                date = pd.to_datetime(line[2:10], format="%Y%m%d")
                expiry = pd.to_datetime(line[202:210], format="%Y%m%d")
                ticker = line[12:24].strip()
                isin = line[230:242].strip()
            except Exception:
                skipped_parse += 1
                continue

            # ── Basic validity ────────────────────────────────────────────────
            if pd.isna(close) or pd.isna(strike) or close <= 0 or strike <= 0:
                skipped_filter += 1
                continue

            # ── DTE filter ────────────────────────────────────────────────────
            dte = (expiry - date).days
            if dte < min_dte or dte > max_dte:
                skipped_filter += 1
                continue

            # ── Liquidity filters ─────────────────────────────────────────────
            if n_trades < min_trades:
                skipped_filter += 1
                continue
            if (not pd.isna(volume)) and volume < min_volume:
                skipped_filter += 1
                continue
            if qty < min_open_interest:
                skipped_filter += 1
                continue
            if close < min_price:
                skipped_filter += 1
                continue

            # ── Bid/ask midpoint (use close as fallback) ──────────────────────
            if (not pd.isna(bid)) and (not pd.isna(ask)) and bid > 0 and ask > 0:
                mid_price = (bid + ask) / 2
                spread = ask - bid
                spread_pct = spread / mid_price if mid_price > 0 else np.nan
            else:
                mid_price = close
                spread = np.nan
                spread_pct = np.nan

            rows.append(
                {
                    "date": date,
                    "ticker": ticker,
                    "underlying": clean_nomres,
                    "isin": isin,
                    "option_type": option_type,
                    "strike": strike,
                    "expiry": expiry,
                    "dte": dte,
                    "close": close,
                    "open": open_,
                    "high": high,
                    "low": low,
                    "avg": avg,
                    "mid_price": mid_price,
                    "bid": bid,
                    "ask": ask,
                    "spread": spread,
                    "spread_pct": spread_pct,
                    "n_trades": n_trades,
                    "qty": qty,
                    "volume_brl": volume,
                }
            )

    if not rows:
        print("No records matched the filters.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = df.sort_values(["date", "underlying", "option_type", "expiry", "strike"])
    df = df.reset_index(drop=True)

    # ── Greeks computation ────────────────────────────────────────────────────
    if compute_greeks_flag:
        if verbose:
            print(f"Computing IV and Greeks for {len(df):,} records...")

        # Merge underlying prices if provided
        if underlying_prices is not None:
            df = df.merge(underlying_prices, on=["date", "underlying"], how="left")
        else:
            df["underlying_price"] = np.nan
            if verbose:
                print(
                    "  WARNING: No underlying prices provided. "
                    "Using avg option price as placeholder — Greeks will be unreliable. "
                    "Pass underlying_prices= for accurate results."
                )

        # Moneyness filter (only when we have underlying price)
        if "underlying_price" in df.columns and df["underlying_price"].notna().any():
            # First: rows where the join found no price are a data problem, not deep OTM
            price_missing = df["underlying_price"].isna()
            if price_missing.any():
                if verbose:
                    missing = sorted(df.loc[price_missing, "underlying"].unique())
                    print(
                        f"  WARNING: {price_missing.sum():,} records dropped — "
                        f"no underlying_price found for: {missing}"
                    )
                df = df[~price_missing].copy()

            # Then: genuine moneyness filter
            moneyness = (df["underlying_price"] - df["strike"]).abs() / df[
                "underlying_price"
            ]
            before = len(df)
            df = df[moneyness <= max_moneyness].copy()
            if verbose and len(df) < before:
                print(
                    f"  Moneyness filter removed {before - len(df):,} deep OTM/ITM records."
                )

        # Merge daily CDI rate; fall back to scalar for any missing dates
        if risk_free_rates is not None:
            df = df.merge(risk_free_rates, on="date", how="left")
            n_missing = df["risk_free_rate"].isna().sum()
            if n_missing > 0:
                if verbose:
                    print(
                        f"  WARNING: {n_missing:,} records have no CDI rate "
                        f"— filling with fallback {risk_free_rate:.4f}"
                    )
                df["risk_free_rate"] = df["risk_free_rate"].fillna(risk_free_rate)
        else:
            df["risk_free_rate"] = risk_free_rate

        def _row_greeks(row):
            S = row.get("underlying_price", np.nan)
            r = row["risk_free_rate"]
            if pd.isna(S) or S <= 0:
                return pd.Series(
                    {
                        "iv": np.nan,
                        "delta": np.nan,
                        "gamma": np.nan,
                        "theta": np.nan,
                        "vega": np.nan,
                    }
                )
            T = row["dte"] / 365.0
            iv = implied_vol(
                row["mid_price"], S, row["strike"], T, r, row["option_type"]
            )
            if pd.isna(iv):
                return pd.Series(
                    {
                        "iv": np.nan,
                        "delta": np.nan,
                        "gamma": np.nan,
                        "theta": np.nan,
                        "vega": np.nan,
                    }
                )
            g = compute_greeks(S, row["strike"], T, r, iv, row["option_type"])
            return pd.Series({"iv": iv, **g})

        greeks_df = df.apply(_row_greeks, axis=1)
        df = pd.concat([df, greeks_df], axis=1)

    if verbose:
        _print_summary(df)

    return df


# ── Summary report ────────────────────────────────────────────────────────────
def _print_summary(df: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("PARSE SUMMARY")
    print("=" * 60)
    print(f"Total records after filters : {len(df):,}")

    if len(df) == 0:
        print("No records to summarize.")
        print("=" * 60)
        return

    print(
        f"Date range                  : {df['date'].min().date()} → {df['date'].max().date()}"
    )
    print(f"Underlyings found           : {sorted(df['underlying'].unique())}")
    print(f"Option types                : {df['option_type'].value_counts().to_dict()}")
    print(f"DTE range                   : {df['dte'].min()} → {df['dte'].max()} days")
    print(
        f"Strike range                : {df['strike'].min():.2f} → {df['strike'].max():.2f}"
    )
    print(
        f"Close price range           : {df['close'].min():.2f} → {df['close'].max():.2f}"
    )

    if "volume_brl" in df.columns:
        print(f"Median daily volume (BRL)   : {df['volume_brl'].median():,.0f}")

    if "spread_pct" in df.columns and df["spread_pct"].notna().any():
        print(f"Median bid/ask spread       : {df['spread_pct'].median():.1%}")
        print(
            f"Records with valid bid/ask  : {df['spread_pct'].notna().sum():,} "
            f"({df['spread_pct'].notna().mean():.1%})"
        )

    if "risk_free_rate" in df.columns:
        print(
            f"Risk-free rate (CDI)        : {df['risk_free_rate'].min():.2%} → "
            f"{df['risk_free_rate'].max():.2%}  (median {df['risk_free_rate'].median():.2%})"
        )

    if "iv" in df.columns and df["iv"].notna().any():
        print(
            f"IV computed successfully    : {df['iv'].notna().sum():,} "
            f"({df['iv'].notna().mean():.1%})"
        )
        print(f"Median IV                   : {df['iv'].median():.1%}")

    print("=" * 60)


# ── Convenience: load equity prices from the same COTAHIST file ───────────────
def extract_equity_prices(
    filepath: str | Path,
    tickers: list[str],
) -> pd.DataFrame:
    """
    Extract daily close prices for equity tickers from the same COTAHIST file.
    Use these as the underlying_prices input to parse_cotahist().

    Returns DataFrame with columns: date, underlying, underlying_price
    """
    filepath = Path(filepath)
    rows = []
    tickers_upper = [t.upper() for t in tickers]

    with open(filepath, encoding="latin-1") as f:
        for line in f:
            if len(line) < 245:
                continue
            if line[0:2] != "01":
                continue
            # BDI 02 = standard lot equities, BDI 14 = ETFs
            if line[10:12].strip() not in ("02", "14"):
                continue
            if line[24:27].strip() != "010":  # TPMERC 010 = spot market
                continue
            ticker = line[12:24].strip().upper()
            if ticker not in tickers_upper:
                continue
            close = _parse_price(line[108:121])
            if pd.isna(close) or close <= 0:
                continue
            rows.append(
                {
                    "date": pd.to_datetime(line[2:10], format="%Y%m%d"),
                    "underlying": line[27:39].strip().upper(),
                    "equity_ticker": ticker,
                    "underlying_price": close,
                }
            )

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["date", "equity_ticker"])
        .sort_values("date")
        .reset_index(drop=True)
    )


# ── Fetch daily CDI rates from BCB ────────────────────────────────────────────
def fetch_cdi_rates(start: str, end: str | None = None) -> pd.DataFrame:
    """
    Fetch daily CDI rates from BCB (series 12) and return as a DataFrame
    with columns ['date', 'risk_free_rate'] where risk_free_rate is the
    annualized rate using the Brazilian 252 business-day convention:
        annualized = (1 + daily_cdi_pct/100)^252 - 1

    Parameters
    ----------
    start : start date string, e.g. '2025-01-01'
    end   : end date string, or None for today

    Returns
    -------
    pd.DataFrame with columns: date (datetime64), risk_free_rate (float)
    """
    raw = sgs.get({"cdi": 12}, start=start, end=end)
    df = raw.reset_index().rename(columns={"Date": "date"})
    df["date"] = pd.to_datetime(df["date"])
    # BCB series 12 gives daily CDI as a percentage (e.g. 0.0486 = 0.0486%/day)
    df["risk_free_rate"] = (1 + df["cdi"] / 100) ** 252 - 1
    return df[["date", "risk_free_rate"]]

In [4]:
FILEPATH = "./data/COTAHIST_A2025.TXT"  # ← update path as needed

# Step 1a: extract underlying equity prices from the same file
# BOVA11 is the iShares Ibovespa ETF; PETR4 is the most liquid single stock
equity_prices = extract_equity_prices(FILEPATH, tickers=["BOVA11", "PETR4"])
print(f"Equity prices extracted: {len(equity_prices)} records")
print(equity_prices.head())

# Step 1b: fetch daily CDI rates from BCB
cdi_rates = fetch_cdi_rates("2025-01-01", "2025-12-31")
print(f"\nCDI rates fetched: {len(cdi_rates)} records")
print(
    f"Annualized CDI range: {cdi_rates['risk_free_rate'].min():.2%} → {cdi_rates['risk_free_rate'].max():.2%}"
)
print(cdi_rates.head())

Equity prices extracted: 500 records
        date    underlying equity_ticker  underlying_price
0 2025-01-02     PETROBRAS         PETR4             36.77
1 2025-01-02  ISHARES BOVA        BOVA11            117.10
2 2025-01-03     PETROBRAS         PETR4             36.38
3 2025-01-03  ISHARES BOVA        BOVA11            115.45
4 2025-01-06     PETROBRAS         PETR4             36.21

CDI rates fetched: 252 records
Annualized CDI range: 12.15% → 14.90%
        date  risk_free_rate
0 2025-01-02          0.1215
1 2025-01-03          0.1215
2 2025-01-06          0.1215
3 2025-01-07          0.1215
4 2025-01-08          0.1215


In [5]:
# Step 2: parse options — run for both to compare liquidity
for name_filter in [["BOVA", "ISHARES"], ["PETR"]]:
    print(f"\n{'='*60}")
    print(f"Parsing options for: {name_filter}")
    # Use lower min_trades for BOVA to capture more options
    min_trades_val = 1 if name_filter == ["BOVA", "ISHARES"] else 10
    df = parse_cotahist(
        FILEPATH,
        underlying_names=name_filter,
        option_types=None,  # both calls and puts
        min_trades=min_trades_val,
        min_volume=1_000,
        min_price=0.01,
        max_moneyness=0.30,
        min_dte=7,
        max_dte=90,
        compute_greeks_flag=True,
        risk_free_rates=cdi_rates,  # daily CDI from BCB
        risk_free_rate=0.1275,  # fallback for any date gaps
        underlying_prices=equity_prices,
        verbose=True,
    )
    if len(df) > 0:
        print("\nSample records:")
        cols = [
            "date",
            "ticker",
            "option_type",
            "strike",
            "expiry",
            "dte",
            "close",
            "n_trades",
            "volume_brl",
            "risk_free_rate",
            "iv",
            "delta",
        ]
        print(df[cols].head(10).to_string(index=False))
        outfile = f"options_{name_filter[0].lower()}.parquet"
        df.to_parquet(outfile, index=False)
        print(f"\nSaved to {outfile}")


Parsing options for: ['BOVA', 'ISHARES']
Parsing COTAHIST_A2025.TXT...
Computing IV and Greeks for 79,875 records...
  Moneyness filter removed 3,364 deep OTM/ITM records.

PARSE SUMMARY
Total records after filters : 76,511
Date range                  : 2025-01-02 → 2025-12-30
Underlyings found           : ['ISHARES BOVA']
Option types                : {'call': 42833, 'put': 33678}
DTE range                   : 7 → 90 days
Strike range                : 82.00 → 205.00
Close price range           : 0.01 → 54.10
Median daily volume (BRL)   : 81,506
Median bid/ask spread       : 30.8%
Records with valid bid/ask  : 38,754 (50.7%)
Risk-free rate (CDI)        : 12.15% → 14.90%  (median 14.90%)
IV computed successfully    : 52,233 (68.3%)
Median IV                   : 17.4%

Sample records:
      date     ticker option_type  strike     expiry  dte  close  n_trades  volume_brl  risk_free_rate       iv    delta
2025-01-02 BOVAA100W2        call   100.0 2025-01-10    8  17.72         1    17720.

## Multi-year backtest data builder

Iterates all `COTAHIST_A*.TXT` files in `./data` and produces two backtest-ready
DataFrames (one for BOVA11, one for PETR4) indexed by date with columns:

| Column | Description |
|---|---|
| `spy_open/high/low/close` | Underlying equity OHLC (BOVA11 or PETR4) |
| `vix_close` | Daily ATM implied volatility — VIX proxy (see below) |
| `risk_free_rate` | Annualized CDI rate from BCB |

### VIX equivalent for B3

**IVBR (Índice de Volatilidade B3)** is the official B3 equivalent of VIX — it uses
IBOVESPA options with the same methodology as CBOE VIX (variance-swap approach across all
strikes). B3 launched it in June 2024, so pre-2024 history does not exist.

**VXEWZ** (CBOE Volatility Index for EWZ, the US-listed Brazil ETF) goes back to 2011
and can be fetched via `yfinance`. It is correlated with BOVA11 vol but reflects USD
hedging costs and EWZ liquidity, so it is a noisy proxy.

**Computed ATM IV** — what we use here. For each trading day we take the
volume-weighted average IV of near-ATM options (moneyness ≤ 5%) in the 20–45 DTE
window. This gives a synthetic ~30-day implied vol index that is fully consistent with
the parsed options data and covers all years.

In [6]:
# ── OHLC extractor ────────────────────────────────────────────────────────────
def extract_equity_ohlc(
    filepath: str | Path,
    tickers: list[str],
) -> pd.DataFrame:
    """
    Extract daily OHLC for equity tickers from a COTAHIST file.
    Returns DataFrame with columns: date, equity_ticker, open, high, low, close
    """
    filepath = Path(filepath)
    rows = []
    tickers_upper = [t.upper() for t in tickers]

    with open(filepath, encoding="latin-1") as f:
        for line in f:
            if len(line) < 245:
                continue
            if line[0:2] != "01":
                continue
            if line[10:12].strip() not in ("02", "14"):
                continue
            if line[24:27].strip() != "010":
                continue
            ticker = line[12:24].strip().upper()
            if ticker not in tickers_upper:
                continue
            open_ = _parse_price(line[56:69])
            high = _parse_price(line[69:82])
            low = _parse_price(line[82:95])
            close = _parse_price(line[108:121])
            if pd.isna(close) or close <= 0:
                continue
            rows.append(
                {
                    "date": pd.to_datetime(line[2:10], format="%Y%m%d"),
                    "equity_ticker": ticker,
                    "open": open_,
                    "high": high,
                    "low": low,
                    "close": close,
                }
            )

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["date", "equity_ticker"])
        .sort_values("date")
        .reset_index(drop=True)
    )


# ── ATM IV proxy (VIX equivalent) ─────────────────────────────────────────────
def compute_daily_atm_iv(
    options_df: pd.DataFrame,
    dte_min: int = 20,
    dte_max: int = 45,
    moneyness_max: float = 0.05,
) -> pd.Series:
    """
    Volume-weighted average IV of near-ATM options in the [dte_min, dte_max] DTE slice.
    Returns a Series indexed by date — used as vix_close.

    Selection criteria per day:
      - dte in [dte_min, dte_max]  →  captures the ~30-day forward vol slice
      - |S - K| / S <= moneyness_max  →  near ATM only (default ±5%)
      - IV is not NaN
    Weights: daily BRL volume (minimum 1 to avoid zero-weight rows).
    """
    if options_df.empty or "iv" not in options_df.columns:
        return pd.Series(dtype=float, name="vix_close")

    mask = (
        (options_df["dte"] >= dte_min)
        & (options_df["dte"] <= dte_max)
        & (
            (options_df["underlying_price"] - options_df["strike"]).abs()
            / options_df["underlying_price"]
            <= moneyness_max
        )
        & options_df["iv"].notna()
    )
    atm = options_df[mask]
    if atm.empty:
        return pd.Series(dtype=float, name="vix_close")

    def wavg(g):
        w = g["volume_brl"].fillna(1.0).clip(lower=1.0)
        return np.average(g["iv"], weights=w)

    return atm.groupby("date").apply(wavg).rename("vix_close")


# ── Multi-year backtest DataFrame builder ─────────────────────────────────────
def build_backtest_df(
    data_dir: str | Path,
    equity_ticker: str,
    underlying_name_filter: list[str],
    cdi_rates: pd.DataFrame,
    min_trades: int = 5,
    dte_min: int = 20,
    dte_max: int = 45,
    atm_moneyness: float = 0.05,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Build a backtest-ready DataFrame from all COTAHIST_A*.TXT files in data_dir.

    For each annual file:
      1. Extracts equity OHLC for equity_ticker
      2. Parses ATM options in [dte_min, dte_max] to compute daily ATM IV (vix_close)
      3. Joins daily CDI rates

    Returns DataFrame indexed by date with columns:
      spy_open, spy_high, spy_low, spy_close, vix_close, risk_free_rate
    """
    data_dir = Path(data_dir)
    files = sorted(data_dir.glob("COTAHIST_A*.TXT"))
    if not files:
        raise FileNotFoundError(f"No COTAHIST_A*.TXT files found in {data_dir}")

    ohlc_parts = []
    opts_parts = []

    for fp in files:
        if verbose:
            print(f"  {fp.name} ...", end=" ", flush=True)

        # Equity OHLC
        ohlc = extract_equity_ohlc(fp, [equity_ticker])
        if not ohlc.empty:
            ohlc_parts.append(ohlc)

        # Equity close prices for BS underlying (same file, left join in parser)
        eq_prices = extract_equity_prices(fp, [equity_ticker])

        # Parse ATM options only — tight DTE window keeps this fast
        opts = parse_cotahist(
            fp,
            underlying_names=underlying_name_filter,
            min_trades=min_trades,
            min_volume=1_000,
            min_price=0.01,
            max_moneyness=0.30,
            min_dte=dte_min,
            max_dte=dte_max,
            compute_greeks_flag=True,
            risk_free_rates=cdi_rates,
            underlying_prices=eq_prices,
            verbose=False,
        )
        if not opts.empty:
            opts_parts.append(opts)

        if verbose:
            n_days = ohlc["date"].nunique() if not ohlc.empty else 0
            print(f"{n_days} trading days, {len(opts):,} ATM option records")

    if not ohlc_parts:
        raise ValueError(f"No OHLC data found for {equity_ticker}.")

    # Combine across years
    ohlc_all = (
        pd.concat(ohlc_parts).drop_duplicates("date").set_index("date").sort_index()
    )

    if opts_parts:
        opts_all = pd.concat(opts_parts, ignore_index=True)
        daily_iv = compute_daily_atm_iv(
            opts_all, dte_min=dte_min, dte_max=dte_max, moneyness_max=atm_moneyness
        )
    else:
        daily_iv = pd.Series(dtype=float, name="vix_close")

    result = pd.DataFrame(
        {
            "spy_open": ohlc_all["open"],
            "spy_high": ohlc_all["high"],
            "spy_low": ohlc_all["low"],
            "spy_close": ohlc_all["close"],
        }
    )
    result = result.join(daily_iv, how="left")
    result = result.join(cdi_rates.set_index("date")[["risk_free_rate"]], how="left")

    if verbose:
        iv_cov = result["vix_close"].notna().mean()
        rfr_cov = result["risk_free_rate"].notna().mean()
        print(
            f"\nResult: {len(result)} trading days"
            f"  |  vix_close coverage {iv_cov:.1%}"
            f"  |  CDI coverage {rfr_cov:.1%}"
        )
        print(result.tail())

    return result

In [7]:
# CDI rates must span all COTAHIST years present in ./data
cdi_all = fetch_cdi_rates("2024-01-01")

print("=" * 60)
print("Building BOVA11 backtest data")
print("=" * 60)
bova_bt = build_backtest_df(
    "./data",
    equity_ticker="BOVA11",
    underlying_name_filter=["BOVA", "ISHARES"],
    cdi_rates=cdi_all,
    min_trades=5,
    dte_min=20,
    dte_max=45,
    atm_moneyness=0.05,
    verbose=True,
)
bova_bt.to_parquet("bova_backtest.parquet")
print("Saved → bova_backtest.parquet")

print()
print("=" * 60)
print("Building PETR4 backtest data")
print("=" * 60)
petr_bt = build_backtest_df(
    "./data",
    equity_ticker="PETR4",
    underlying_name_filter=["PETR"],
    cdi_rates=cdi_all,
    min_trades=5,
    dte_min=20,
    dte_max=45,
    atm_moneyness=0.05,
    verbose=True,
)
petr_bt.to_parquet("petr_backtest.parquet")
print("Saved → petr_backtest.parquet")

Building BOVA11 backtest data
  COTAHIST_A2024.TXT ... 251 trading days, 15,959 ATM option records
  COTAHIST_A2025.TXT ... 250 trading days, 21,010 ATM option records
  COTAHIST_A2026.TXT ... 81 trading days, 9,856 ATM option records

Result: 582 trading days  |  vix_close coverage 98.5%  |  CDI coverage 100.0%
            spy_open  spy_high  spy_low  spy_close  vix_close  risk_free_rate
date                                                                         
2026-04-24    188.10    188.13   186.52     187.01   0.196418        0.146499
2026-04-27    187.56    187.85   186.06     186.06        NaN        0.146499
2026-04-28    184.13    185.57   183.94     185.16   0.183707        0.146499
2026-04-29    184.00    184.21   181.03     181.03   0.188965        0.146499
2026-04-30    182.78    184.66   182.49     184.09   0.194533        0.144001
Saved → bova_backtest.parquet

Building PETR4 backtest data
  COTAHIST_A2024.TXT ... 251 trading days, 28,220 ATM option records
  COTAHIST_

In [9]:
bova_bt.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 582 entries, 2024-01-02 to 2026-04-30
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   spy_open        582 non-null    float64
 1   spy_high        582 non-null    float64
 2   spy_low         582 non-null    float64
 3   spy_close       582 non-null    float64
 4   vix_close       573 non-null    float64
 5   risk_free_rate  582 non-null    float64
dtypes: float64(6)
memory usage: 31.8 KB


In [12]:
bova_bt.describe().round(2)

,spy_open,spy_high,spy_low,spy_close,vix_close,risk_free_rate
count,582.00,582.00,582.00,582.00,573.00,582.00
mean,136.61,137.47,135.84,136.65,0.16,0.13
std,19.55,19.81,19.36,19.61,0.03,0.02
min,115.74,116.24,115.23,115.45,0.10,0.10
25%,123.94,124.59,123.28,123.88,0.14,0.11
50%,129.04,129.82,128.42,128.99,0.15,0.13
75%,139.96,141.10,139.56,140.44,0.16,0.15
max,194.80,195.73,194.40,195.16,0.51,0.15
